In [2]:
from ta.volatility import AverageTrueRange

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from backtesting import Strategy
from backtesting import Backtest, set_bokeh_output
set_bokeh_output(notebook=False)

from datetime import timedelta

import sys
sys.path.append('../utils')
import Util
import importlib
importlib.reload(Util)

from enum import Enum
import time

from random import randint
import random


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [3]:
pair = 'GBPJPY'

df_d1_raw = pd.read_csv(f"../Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h4_raw = pd.read_csv(f"../Trading Data/{pair}_H4.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_m15_raw = pd.read_csv(f"../Trading Data/{pair}_M15.csv",sep=',', parse_dates=['Date'], index_col='Date')

print(df_d1_raw)
print(df_h4_raw)
print(df_m15_raw)

               Open     High      Low    Close  Tickvol
Date                                                   
2010-09-10  129.605  130.065  128.995  129.255   172229
2010-09-13  129.565  130.115  128.780  129.090   215859
2010-09-14  129.105  129.430  127.640  129.015   226170
2010-09-15  129.015  134.070  128.555  133.975   252317
2010-09-16  133.975  134.260  132.970  134.150   222443
...             ...      ...      ...      ...      ...
2026-09-18  208.323  211.276  208.233  210.095   364707
2026-09-21  210.296  210.762  209.676  210.321   243731
2026-09-22  210.321  210.892  209.513  210.080   333200
2026-09-23  210.080  210.176  209.422  209.564   273936
2026-09-24  209.566  210.105  208.768  209.848   199988

[4183 rows x 5 columns]
                        Open     High      Low    Close  Tickvol
Date                                                            
2010-09-10 04:00:00  129.605  129.630  128.995  129.385    25539
2010-09-10 08:00:00  129.380  129.985  129.275  129.

In [4]:
structure_d1_window = 2
structure_m15_window = 2

date_format = '%Y-%m-%d %H:%M'
# start_date = '2023-1-1 00:00'
# end_date = '2024-3-29 00:00'
start_date = datetime.strptime('2021-1-1 00:00', date_format)
end_date = datetime.strptime('2026-9-16 00:00', date_format)
# start_date = '2022-01-1 00:00'

timezone = "Europe/London"

# nombre de jours pour les high.low
define_key_level_window = 90

gap_look_ahead = 6

# nombre de candle au dessus/ en desous de key level après breakout
breakout_candles = 9

# nombre de jours à chercher key level
key_level_lookback_candle = 100

# vérifier si 30 candle dans ke passé pour valider un valid breakout,
# un valid breakout doit avoir {breakout_candles} candles au dessus du resistant level
breakout_candles_lookback = 30

start_trade_hour = 9
stop_trade_hour = 19

atr_multiplier = 0.7
min_touched = 3

In [5]:
# process D1 data
df_d1 = df_d1_raw.loc[start_date - timedelta(days=30) : end_date]

df_d1 = Util.implement_market_structure_states_fixed_pivots(df_d1, n=structure_d1_window)
df_d1["state_changed"] = df_d1["market_state"] != df_d1[
    "market_state"
].shift(1)
for state in df_d1[df_d1["state_changed"]][["Close", "market_state"]].iterrows():
    print(state)
df_d1 = df_d1.loc[start_date : end_date]

(Timestamp('2020-12-02 00:00:00'), Close           139.632
market_state        NaN
Name: 2020-12-02 00:00:00, dtype: object)
(Timestamp('2020-12-03 00:00:00'), Close           139.68
market_state       NaN
Name: 2020-12-03 00:00:00, dtype: object)
(Timestamp('2020-12-04 00:00:00'), Close           139.937
market_state        NaN
Name: 2020-12-04 00:00:00, dtype: object)
(Timestamp('2020-12-07 00:00:00'), Close           139.114
market_state        NaN
Name: 2020-12-07 00:00:00, dtype: object)
(Timestamp('2020-12-08 00:00:00'), Close           139.134
market_state        NaN
Name: 2020-12-08 00:00:00, dtype: object)
(Timestamp('2020-12-09 00:00:00'), Close           139.302
market_state        NaN
Name: 2020-12-09 00:00:00, dtype: object)
(Timestamp('2020-12-10 00:00:00'), Close           138.634
market_state        NaN
Name: 2020-12-10 00:00:00, dtype: object)
(Timestamp('2020-12-11 00:00:00'), Close           137.506
market_state    BEARISH
Name: 2020-12-11 00:00:00, dtype: object)
(T

In [6]:
# process h4 data
df_h4 = df_h4_raw.loc[start_date - timedelta(days=20) : end_date]
df_h4 = Util.implement_market_structure_states_fixed_pivots(df_h4, 5)
df_h4 = df_h4.loc[start_date : end_date]

h4_d1_direction_list = []
for i in range(len(df_d1)):
    row = df_d1.iloc[i]
    start_date_daily = row.name
    end_date_daily = start_date_daily + timedelta(hours=23, minutes=59)
    df_by_day = df_h4.loc[f'{start_date_daily}' : f'{end_date_daily}']

    for index2, row2 in df_by_day.iterrows():
        h4_d1_direction_list.append(df_d1.iloc[i-1]['market_state'])

df_h4['market_state_d1'] = h4_d1_direction_list

indicator_atr_14 = AverageTrueRange(df_h4['High'], df_h4['Low'], df_h4['Close'], window=14)
df_h4['ATR'] = indicator_atr_14._atr

df_h4

,Open,High,Low,Close,Tickvol,is_swing_high,is_swing_low,market_state,market_state_d1,ATR
Date,,,,,,,,,,
2021-01-04 00:00:00,140.944,141.304,140.776,141.005,37352,True,False,BULLISH,BEARISH,0.000000
2021-01-04 04:00:00,141.007,141.043,140.847,140.938,20550,False,False,BULLISH,BEARISH,0.000000
2021-01-04 08:00:00,140.936,141.064,140.412,140.757,47343,False,False,BULLISH,BEARISH,0.000000
2021-01-04 12:00:00,140.758,140.861,140.188,140.226,55445,False,False,BULLISH,BEARISH,0.000000
2021-01-04 16:00:00,140.227,140.233,139.687,139.870,55631,False,False,BULLISH,BEARISH,0.000000
...,...,...,...,...,...,...,...,...,...,...
2026-09-15 08:00:00,208.652,209.090,208.514,208.655,56407,False,False,BEARISH,BEARISH,0.567701
2026-09-15 12:00:00,208.655,209.211,208.607,209.168,74480,False,False,BULLISH,BEARISH,0.570294
2026-09-15 16:00:00,209.167,209.224,208.885,209.066,38410,False,False,BULLISH,BEARISH,0.553773


In [7]:
# process m15 data
df_m15 = df_m15_raw.loc[start_date - timedelta(hours=3): end_date]
df_m15 = Util.implement_market_structure_states_fixed_pivots(df_m15, n=structure_m15_window)

m15_daily_direction_list = []

index_list = []

df_m15 = df_m15.loc[start_date : end_date]
for i in range(len(df_h4)):
    row = df_h4.iloc[i]
    start_date_h4 = row.name
    end_date_h4 = start_date_h4 + timedelta(hours=3, minutes=59)
    df_by_day = df_m15.loc[f'{start_date_h4}' : f'{end_date_h4}']

    for index2, row2 in df_by_day.iterrows():
        # affect current market states of H4 candle because the result is market state of previous D1 candle
        m15_daily_direction_list.append(df_h4.iloc[i]['market_state_d1'])

        # ajust to daylight saving time
        dst = pd.Timestamp(index2, tz=timezone).dst()
        if dst == timedelta(0):
            index2 = index2 - timedelta(hours=1)
            index_list.append(index2)
        else:
            index_list.append(index2)

df_m15['market_state_d1'] = m15_daily_direction_list

df_m15.index = index_list

df_m15

,Open,High,Low,Close,Tickvol,is_swing_high,is_swing_low,market_state,market_state_d1
2021-01-03 23:00:00,140.944,141.110,140.944,141.100,2767,False,True,NaN,BEARISH
2021-01-03 23:15:00,141.099,141.204,141.070,141.186,1223,False,False,NaN,BEARISH
2021-01-03 23:30:00,141.186,141.304,141.158,141.282,1042,True,False,NaN,BEARISH
2021-01-03 23:45:00,141.278,141.301,141.218,141.244,1886,False,False,NaN,BEARISH
2021-01-04 00:00:00,141.247,141.275,140.988,141.007,3577,False,False,BEARISH,BEARISH
...,...,...,...,...,...,...,...,...,...
2026-09-15 23:00:00,208.895,209.029,208.895,209.018,1853,True,False,BEARISH,BEARISH
2026-09-15 23:15:00,209.020,209.027,208.982,208.996,1268,False,False,BEARISH,BEARISH
2026-09-15 23:30:00,208.997,209.013,208.926,208.953,1279,False,False,BEARISH,BEARISH
2026-09-15 23:45:00,208.951,209.021,208.923,209.019,1261,False,False,BEARISH,BEARISH


In [8]:
# using m15 market structure for entry
def apply_total_signal(df, start_trade_hour=9, stop_trade_hour=19):
    # Initialize the 'TotalSignal' column
    df_signal_list = []

    for i in range(0, len(df)):
        current_bar_date = df.index[i]
        signal = 0

        # short
        # d1 trending conditions
        c_short_daily_direction = df['market_state_d1'].iloc[i-1] == 'BEARISH'
        if c_short_daily_direction:
            c_m15_ms_bearish = df['market_state'].iloc[i] == 'BEARISH'

            c_time_trade = start_trade_hour <= current_bar_date.hour and current_bar_date.hour <= stop_trade_hour
            if c_time_trade and c_m15_ms_bearish:
                signal = 1

        # long
        c_long_daily_direction = df['market_state_d1'].iloc[i-1] == 'BULLISH'
        if c_long_daily_direction:
            c_m15_ms_bullish = df['market_state'].iloc[i] == 'BULLISH'

            c_time_trade = start_trade_hour <= current_bar_date.hour and current_bar_date.hour <= stop_trade_hour
            if c_time_trade and c_m15_ms_bullish:
                signal = 2

        df_signal_list.append(signal)

    return df_signal_list

df_m15['TotalSignal'] = apply_total_signal(df=df_m15, start_trade_hour=9, stop_trade_hour=19)

In [9]:
len(df_m15[df_m15.TotalSignal != 0])

32481

In [10]:
df_m15['pointpos'] = df_m15.apply(lambda row: Util.pointpos(row), axis=1)

In [11]:
start_date_chart = '2022-10-1 00:00'
end_date_chart = '2022-10-30 00:00'
dfpl = df_m15.loc[start_date_chart : end_date_chart]#.set_index("Gmt time")

# Create a plot with 2 rows
fig = make_subplots(rows=1, cols=1)

# Add candlestick plot on the first row
fig.add_trace(go.Candlestick(x=dfpl.index,
                             open=dfpl['Open'],
                             high=dfpl['High'],
                             low=dfpl['Low'],
                             close=dfpl['Close']),
              row=1, col=1)

# Add markers for entry point m15
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['pointpos'], mode="markers",
                         marker=dict(size=12, color="MediumPurple"),
                         name="entry"),
              row=1, col=1)

fig.update_layout(width=1500, height=800, sliders=[])
fig.show()

In [12]:
class ChochStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    setup_duration_gap = 60
    stoploss_window = 150
    
    def init(self):
        self.current_day = None
        self.setup_duration = None
        self.order_placed_time = None

        super().init()

    def reset_order(self, current_bar_date):
        self.current_day = current_bar_date

        # delete placed order at the end of the day if not executed
        for order in self.orders:
            order_executed= False
            for trade in self.trades:
                if trade.tag == order.tag:
                    order_executed = True

            if order_executed == False:
                order.cancel()

        self.order_placed_time = None
        
    def get_tag(self):
        return f"{self.order_placed_time}"

    def get_highest_high(self):
            return self.data.High[len(self.data.High) - self.stoploss_window : len(self.data.High)].max()
        
    def get_lowest_low(self):
        return self.data.Low[len(self.data.Low) - self.stoploss_window : len(self.data.Low)].min()
    
    def next(self):
        super().next()

        current_bar_date = self.data.index[-1].date()
        if self.current_day != current_bar_date:
            self.reset_order(current_bar_date)

        if len(self.trades)==0:
            if self.data['TotalSignal'][-1]==1:
                planned_entry_price = self.data.Close[-1]
                sl = abs(planned_entry_price - self.get_highest_high())
                if sl == 0:
                    sl = round(random.uniform(0.3,1), 1)
                tp = self.TPcoef * sl

                sl1 = planned_entry_price + sl
                tp1 = planned_entry_price - tp
                position_size = int((self.risk_percent * self.equity) / sl)

                self.sell(sl=sl1, tp=tp1, size=position_size)
                
            if self.data['TotalSignal'][-1]==2:
                planned_entry_price = self.data.Close[-1]
                sl = abs(planned_entry_price - self.get_lowest_low())
                if sl == 0:
                    sl = round(random.uniform(0.3,1), 1)
                tp = self.TPcoef * sl

                sl1 = planned_entry_price - sl
                tp1 = planned_entry_price + tp
                position_size = int((self.risk_percent * self.equity) / sl)

                self.buy(sl=sl1, tp=tp1, size=position_size)

In [13]:
start_date_chart = '2021-1-1 00:00'
end_date_chart = '2026-12-30 00:00'
global_dfopt = df_m15.loc[start_date_chart : end_date_chart]

bt = Backtest(global_dfopt, ChochStrat, cash=10000, margin=0.005, finalize_trades=True) #0.0002
stats = bt.run()

print(stats)
bt.plot()


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=14076: Broker canceled the order due to insufficient margin (equity=9025.54, margin_available=9025.54).
  warnings.warn(


Start                     2021-01-03 23:00:00
End                       2026-09-16 00:00:00
Duration                   2081 days 01:00:00
Exposure Time [%]                    93.33804
Equity Final [$]                     9290.274
Equity Peak [$]                     11510.695
Return [%]                           -7.09726
Buy & Hold Return [%]                48.17505
Return (Ann.) [%]                     -1.1492
Volatility (Ann.) [%]                  9.3691
CAGR [%]                             -1.28376
Sharpe Ratio                         -0.12266
Sortino Ratio                        -0.16811
Calmar Ratio                         -0.04655
Alpha [%]                           -10.82243
Beta                                  0.07733
Max. Drawdown [%]                   -24.68731
Avg. Drawdown [%]                    -0.67535
Max. Drawdown Duration     1235 days 16:00:00
Avg. Drawdown Duration       19 days 22:58:00
# Trades                                  289
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


GridPlot(id='p1261', ...)

In [14]:
# add some sl tp management

class RandomStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    trades_list = None
    stoploss_window = 150

    def is_signal(self, date):
        for i in range(0, len(self.trades_list)):
            if self.trades_list[i] == date:
                return True
        return False

    def get_highest_high(self):
        return self.data.High[len(self.data.High) - self.stoploss_window : len(self.data.High)].max()
    
    def get_lowest_low(self):
        return self.data.Low[len(self.data.Low) - self.stoploss_window : len(self.data.Low)].min()
    
    def init(self):
        super().init()

    def next(self):
        super().next()
        if self.is_signal(self.data.index[-1]) and len(self.trades) == 0 and len(self.closed_trades) < 289:
            if self.data['market_state_d1'][-1] == 'BULLISH':
                planned_entry_price = self.data.Close[-1]
                sl = abs(planned_entry_price - self.get_lowest_low())
                if sl == 0:
                    sl = round(random.uniform(0.3,1), 1)
                tp = self.TPcoef * sl

                sl1 = planned_entry_price - sl
                tp1 = planned_entry_price + tp
                position_size = int((self.risk_percent * self.equity) / sl)
                # max_affordable_size = int((self.equity * 0.99) / planned_entry_price)
                # print(f'{position_size} {max_affordable_size}')

                self.buy(sl=sl1, tp=tp1, size=position_size)
            else:
                planned_entry_price = self.data.Close[-1]
                sl = abs(planned_entry_price - self.get_highest_high())
                if sl == 0:
                    sl = round(random.uniform(0.3,1), 1)
                tp = self.TPcoef * sl

                sl1 = planned_entry_price + sl
                tp1 = planned_entry_price - tp
                position_size = int((self.risk_percent * self.equity) / sl)
                # max_affordable_size = int((self.equity * 0.99) / planned_entry_price)
                # print(f'{position_size} {max_affordable_size}')

                self.sell(sl=sl1, tp=tp1, size=position_size)

In [17]:
trades_list = []
while len(trades_list) != 700:
    random_index = randint(0, len(df_m15))
    candle = df_m15.iloc[random_index]
    # exclu Saturday and Sunday
    if candle.name.weekday() < 5:
        if 9 <= candle.name.hour and candle.name.hour <= 19:
            trades_list.append(candle.name)
trades_list.sort()

bt = Backtest(df_m15, RandomStrat, cash=10000, margin=0.005, finalize_trades=True) #0.0002
stats = bt.run(trades_list=trades_list)

print(stats)

bt.plot()

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=29538: Broker canceled the order due to insufficient margin (equity=9760.62, margin_available=9760.62).
  warnings.warn(
c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=113238: Broker canceled the order due to insufficient margin (equity=11504.64, margin_available=11504.64).
  warnings.warn(


Start                     2021-01-03 23:00:00
End                       2026-09-16 00:00:00
Duration                   2081 days 01:00:00
Exposure Time [%]                    65.78492
Equity Final [$]                    10170.746
Equity Peak [$]                     11739.798
Return [%]                            1.70746
Buy & Hold Return [%]                48.17505
Return (Ann.) [%]                     0.26618
Volatility (Ann.) [%]                 8.03642
CAGR [%]                              0.29759
Sharpe Ratio                          0.03312
Sortino Ratio                         0.04858
Calmar Ratio                           0.0178
Alpha [%]                            -0.42501
Beta                                  0.04426
Max. Drawdown [%]                   -14.95481
Avg. Drawdown [%]                    -0.50626
Max. Drawdown Duration     1285 days 09:00:00
Avg. Drawdown Duration       15 days 14:31:00
# Trades                                  221
Win Rate [%]                      

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '8h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


GridPlot(id='p1829', ...)

In [19]:
i = 0
expectancy_list = []
winrate_list = []
number_trades = []

while len(expectancy_list) < 500:
    trades_list = []
    while len(trades_list) != 700:
        random_index = randint(0, len(df_m15)-1)
        candle = df_m15.iloc[random_index -1]
        # exclu Saturday and Sunday
        if candle.name.weekday() < 5:
            if 9 <= candle.name.hour and candle.name.hour <= 19:
                trades_list.append(candle.name)
    trades_list.sort()

    bt = Backtest(df_m15, RandomStrat, cash=10000, margin=0.0009, finalize_trades=True) #0.0002
    stats = bt.run(trades_list=trades_list)
    if 200 <= stats['# Trades'] and stats['# Trades'] <= 340:
        expectancy_list.append(stats['Expectancy [%]'])
        winrate_list.append(stats['Win Rate [%]'])
        number_trades.append(stats['# Trades'])
        print(f'{i} {len(expectancy_list)} {stats['Expectancy [%]']} Done')
    i += 1

print(expectancy_list)

0 1 -0.05025464773836021 Done


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=26794: Broker canceled the order due to insufficient margin (equity=10152.36, margin_available=10152.36).
  warnings.warn(


1 2 -0.04287313200611774 Done
2 3 -0.015322464792428311 Done
3 4 -0.05961390139549315 Done
4 5 -0.004706843416911728 Done
5 6 -0.01878648361787807 Done
6 7 0.010241153841035898 Done
7 8 -0.03589121806386217 Done
8 9 -0.088136280835197 Done
9 10 -0.05285384835711022 Done
10 11 0.0290503747680437 Done
11 12 -0.06952464372535219 Done
12 13 0.052920744875391786 Done


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=35667: Broker canceled the order due to insufficient margin (equity=8926.71, margin_available=8926.71).
  warnings.warn(


13 14 -0.060876652751879846 Done
14 15 0.01574338341644563 Done
15 16 -0.09817417441233599 Done


...

469 468 -0.06859532075808915 Done
470 469 -0.07236398689041404 Done
471 470 0.0031889936933231127 Done
472 471 -0.05062739870863091 Done
473 472 -0.017399640385308855 Done
474 473 -0.024016142231002985 Done


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=96933: Broker canceled the order due to insufficient margin (equity=10179.94, margin_available=10179.94).
  warnings.warn(


475 474 -0.08168349643110404 Done
476 475 -0.025258982776639818 Done
477 476 0.006470677718272269 Done
478 477 0.010572182239893318 Done
479 478 -0.01860299037583771 Done
480 479 -0.06096505839437115 Done
481 480 -0.08364841581399188 Done
482 481 -0.06246184401783331 Done
483 482 0.015843658137004137 Done
484 483 -0.01605707403071941 Done
485 484 -0.03922757786361558 Done
486 485 -0.07478770712345637 Done


c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\backtesting.py:1009: UserWarning: time=9256: Broker canceled the order due to insufficient margin (equity=10908.81, margin_available=10908.81).
  warnings.warn(


487 486 -0.01748507851646568 Done
488 487 -0.03239719501864659 Done
489 488 -0.13839161133353745 Done
490 489 0.03325447677809094 Done
491 490 -0.013043871522379263 Done
492 491 -0.0023090736446622403 Done
493 492 -0.02439984822525093 Done
494 493 0.07853308542329857 Done
495 494 0.0023470228361754083 Done
496 495 -0.029549757552540877 Done
497 496 0.011700941822724952 Done
498 497 -0.05834014071669797 Done
499 498 -0.10403730358633162 Done
500 499 -0.042435373930993045 Done
501 500 -0.0710083377265901 Done
[np.float64(-0.05025464773836021), np.float64(-0.04287313200611774), np.float64(-0.015322464792428311), np.float64(-0.05961390139549315), np.float64(-0.004706843416911728), np.float64(-0.01878648361787807), np.float64(0.010241153841035898), np.float64(-0.03589121806386217), np.float64(-0.088136280835197), np.float64(-0.05285384835711022), np.float64(0.0290503747680437), np.float64(-0.06952464372535219), np.float64(0.052920744875391786), np.float64(-0.060876652751879846), np.float64(

In [20]:
print(len(expectancy_list))

p95 = np.percentile(expectancy_list, 95)                  # ngưỡng phân vị 95
print(f"p95 = {p95}")
# max_expectancy = max(expectancy_list_2)
print(f"max = {max(expectancy_list)}")
mean_expectancy = np.mean(expectancy_list)
print(f"mean = {mean_expectancy}")
print(f"min = {min(expectancy_list)}")
print(f"gap = {-0.06797 - mean_expectancy}")

df_result = pd.DataFrame({'expectancy' : expectancy_list})
rank = (df_result['expectancy'] <= -0.06797).mean() * 100 
print(f"rank = {rank}")
p_value = ((df_result['expectancy'] >= -0.06797).sum() + 1) / (len(df_result['expectancy']) + 1)
print(f"p_value = {p_value}")

print(len(winrate_list))

p95_winrate = np.percentile(winrate_list, 95)                  # ngưỡng phân vị 95
print(f"p95 = {p95_winrate}")
# max_expectancy = max(expectancy_list_2)
print(f"max = {max(winrate_list)}")
mean_winrate = np.mean(winrate_list)
print(f"mean = {mean_winrate}")
print(f"min = {min(winrate_list)}")
print(f"gap = {33.21799 - mean_winrate}")

500
p95 = 0.02980341136542391
max = 0.10963342416583462
mean = -0.04596080161428236
min = -0.19222577636834565
gap = -0.022009198385717646
rank = 32.4
p_value = 0.6766467065868264
500
p95 = 37.04439096135847
max = 40.74074074074074
mean = 33.305330064260744
min = 26.209677419354836
gap = -0.08734006426074359
